# 04 — Raw/v2 Training Data Preparation

Prepare or verify the frozen five-class S10 Raw/Gamma-or-Skip pair and shared Raw development input. Reuse byte-equivalent exports; do not select a new policy, resplit data or train a detector.

- **Inputs:** Frozen protocol, subsets, v2 policy, original BDD sources and saved six-class S10/development labels. Existing five-class exports are reused after deep verification.
- **Outputs:** `docs/round2-training-v1/preparation.json` and compressed per-image routes; datasets stay under the configured gitignored preparation root.
- **Prerequisites:** `uv sync --locked --extra study` and the saved local inputs listed in section 1.

**Outline**

1. [Configuration and Prerequisites](#configuration-prerequisites)
2. [Matched Inputs and Original-Pixel Verification](#matched-inputs)
3. [Route and Source Manifest](#route-manifest)
4. [Preparation Summary and P5 Handoff](#preparation-summary)

Set paths in section 1, then **Restart & Run All**. This notebook executes only its own phase. Missing or stale inputs fail; original data, policies, historical evidence and final-holdout pixels remain untouched.

<a id="configuration-prerequisites"></a>

## 1. Configuration and Prerequisites

The defaults reuse existing exports. To create missing exports, select a new `PREPARED` and report root; existing directories are audited and never repaired in place.

In [ ]:
from pathlib import Path

import cv2
import pandas as pd
from IPython.display import display

from vehicle_detection.common import freeze_json, read_json, require_paths
from vehicle_detection.training_preparation import prepare_training_data

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
PROTOCOL_PATH = ROOT / "runs/issue5-v1/protocol.json"
SUBSETS_PATH = ROOT / "runs/issue6-v2/subsets.json"
POLICY_PATH = ROOT / "docs/issue6-v2/policy.json"
DATA_ROOT = ROOT / "data/bdd100k"
TRAIN_CONVERSION = ROOT / "runs/issue6-v2/conversion/S10"
DEV_CONVERSION = ROOT / "runs/issue5-v1/conversion"
PREPARED = ROOT / "runs/preprocessing-five-v1"
REPORT_ROOT = ROOT / "docs/round2-training-v1"
require_paths(
    protocol=PROTOCOL_PATH,
    subsets=SUBSETS_PATH,
    policy=POLICY_PATH,
    sources=DATA_ROOT,
    training_labels=TRAIN_CONVERSION / "conversion.json",
    development_labels=DEV_CONVERSION / "conversion.json",
)
cv2.setNumThreads(1)

<a id="matched-inputs"></a>

## 2. Matched Inputs and Original-Pixel Verification

Verify exact frozen roles, five-class labels/geometry, paired JPEG95 encoding and source hashes. Replay the frozen recipe from original pixels and compare every output hash and actual route. Only S10 and shared Raw development pixels are read; no AP or detector is used.

In [ ]:
preparation, routes = prepare_training_data(
    read_json(PROTOCOL_PATH),
    read_json(SUBSETS_PATH),
    read_json(POLICY_PATH),
    DATA_ROOT,
    PREPARED,
    TRAIN_CONVERSION,
    DEV_CONVERSION,
)
print(
    "Verified images:",
    {name: len(export["ids"]) for name, export in preparation["exports"].items()},
)
print("Frozen policy:", preparation["policy_id"])

<a id="route-manifest"></a>

## 3. Route and Source Manifest

Retain one row per S10 image with source group, input/output/label hashes and the actual v2 action. These counts describe execution, not detector improvement.

In [ ]:
import gzip

from vehicle_detection.common import file_hash

REPORT_ROOT.mkdir(parents=True, exist_ok=True)
route_path = REPORT_ROOT / "routes.csv.gz"
compressed = gzip.compress(pd.DataFrame(routes).to_csv(index=False).encode(), mtime=0)
if route_path.exists() and route_path.read_bytes() != compressed:
    raise ValueError("Route evidence changed; choose a new report root")
if not route_path.exists():
    route_path.write_bytes(compressed)
display(pd.Series(preparation["route_counts"], name="S10 images"))
print("Route manifest SHA256:", file_hash(route_path))

<a id="preparation-summary"></a>

## 4. Preparation Summary and P5 Handoff

The P5 notebook reads this immutable receipt without rerunning P4. Changes to pixels, policy, classes or protocol need a fresh preparation/report root. S20 is outside this default study.

In [ ]:
freeze_json(REPORT_ROOT / "preparation.json", preparation)
print("P4 identity:", preparation["preparation_id"])
print("Next: independently run 05_model_training.ipynb")
print("No training, policy tuning, confirmation or final-test access occurred.")